In [ ]:
# In[1]:

import pandas as pd
import pytz

# Timezone for all displays
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into DataFrames (stored for future steps)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to create compact overview for a dataframe with timestamp & cmdb_id
def make_overview(df, ts_col='timestamp', cmdb_col='cmdb_id'):
    # Ensure timestamp exists and is numeric
    if ts_col not in df.columns or df.empty:
        overview = {
            'rows': len(df),
            'min_timestamp_epoch': None,
            'min_timestamp_local': None,
            'max_timestamp_epoch': None,
            'max_timestamp_local': None,
            'unique_cmdb_ids_sample': []
        }
        return pd.DataFrame([overview])
    # Coerce timestamps to numeric
    df_ts = df.copy()
    df_ts[ts_col] = pd.to_numeric(df_ts[ts_col], errors='coerce').astype('Int64')
    # Compute min/max epochs (skip NA)
    min_epoch = int(df_ts[ts_col].dropna().min()) if df_ts[ts_col].dropna().size>0 else None
    max_epoch = int(df_ts[ts_col].dropna().max()) if df_ts[ts_col].dropna().size>0 else None
    # Convert to localized datetimes if available
    min_local = (pd.to_datetime(min_epoch, unit='s', utc=True).tz_convert(tz)
                 if min_epoch is not None else None)
    max_local = (pd.to_datetime(max_epoch, unit='s', utc=True).tz_convert(tz)
                 if max_epoch is not None else None)
    # Unique cmdb ids sample (limit 50)
    if cmdb_col in df_ts.columns:
        unique_cmdb = df_ts[cmdb_col].dropna().astype(str).unique()
        unique_cmdb_sample = list(unique_cmdb[:50])
    else:
        unique_cmdb_sample = []
    overview = {
        'rows': len(df),
        'min_timestamp_epoch': min_epoch,
        'min_timestamp_local': min_local.isoformat() if min_local is not None else None,
        'max_timestamp_epoch': max_epoch,
        'max_timestamp_local': max_local.isoformat() if max_local is not None else None,
        'unique_cmdb_ids_sample': unique_cmdb_sample
    }
    return pd.DataFrame([overview])

# Create overviews
metric_overview = make_overview(metric_df, ts_col='timestamp', cmdb_col='cmdb_id')
trace_overview = make_overview(trace_df, ts_col='timestamp', cmdb_col='cmdb_id')
log_overview = make_overview(log_df, ts_col='timestamp', cmdb_col='cmdb_id')
error_overview = make_overview(error_df, ts_col='timestamp', cmdb_col='cmdb_id')

# KPI/name summaries (top counts)
if 'kpi_name' in metric_df.columns:
    metric_kpi_counts = (metric_df.groupby('kpi_name', dropna=False)
                         .size().reset_index(name='count')
                         .sort_values('count', ascending=False)
                         .head(20)
                         .reset_index(drop=True))
else:
    metric_kpi_counts = pd.DataFrame(columns=['kpi_name','count'])

if 'trace_name' in trace_df.columns:
    trace_name_counts = (trace_df.groupby('trace_name', dropna=False)
                         .size().reset_index(name='count')
                         .sort_values('count', ascending=False)
                         .head(20)
                         .reset_index(drop=True))
else:
    trace_name_counts = pd.DataFrame(columns=['trace_name','count'])

if 'log_name' in log_df.columns:
    log_name_counts = (log_df.groupby('log_name', dropna=False)
                       .size().reset_index(name='count')
                       .sort_values('count', ascending=False)
                       .head(20)
                       .reset_index(drop=True))
else:
    log_name_counts = pd.DataFrame(columns=['log_name','count'])

# error_logs: top 10 most frequent message snippets (group by message)
if 'message' in error_df.columns and not error_df.empty:
    error_message_counts = (error_df.groupby('message', dropna=False)
                            .size().reset_index(name='count')
                            .sort_values('count', ascending=False)
                            .head(10)
                            .reset_index(drop=True))
else:
    error_message_counts = pd.DataFrame(columns=['message','count'])

# For each cmdb_id in error_df show earliest timestamp and total message count (limit to first 20)
if 'cmdb_id' in error_df.columns and 'timestamp' in error_df.columns and not error_df.empty:
    # ensure timestamp numeric
    edf = error_df.copy()
    edf['timestamp'] = pd.to_numeric(edf['timestamp'], errors='coerce').astype('Int64')
    grp = edf.groupby('cmdb_id', dropna=False)['timestamp'].agg(['min','count']).reset_index()
    # convert min epoch to localized isoformat
    def epoch_to_local_iso(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat()
    grp['earliest_timestamp_local'] = grp['min'].apply(lambda x: epoch_to_local_iso(x))
    error_cmdb_summary = grp.rename(columns={'count':'message_count','cmdb_id':'cmdb_id'}) \
                           [['cmdb_id','earliest_timestamp_local','message_count']] \
                           .sort_values('message_count', ascending=False) \
                           .head(20) \
                           .reset_index(drop=True)
else:
    error_cmdb_summary = pd.DataFrame(columns=['cmdb_id','earliest_timestamp_local','message_count'])

# Keep final displayed outputs compact.
# Variables to display (DataFrames / summaries)
metric_overview, metric_kpi_counts, trace_overview, trace_name_counts, log_overview, log_name_counts, error_overview, error_message_counts, error_cmdb_summary

```
Out[1]:
```
```python
# Summary of telemetry file loads (plain English)
summary = (
    "Summary of loaded telemetry files (timestamps shown in UTC+8):\n\n"
    "1) metric.csv\n"
    "- Total rows: 28,061\n"
    "- Timestamp range: 2021-03-04 07:00:00 to 2021-03-04 07:30:00\n"
    "- Sample of unique cmdb_id values: IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis... (first 50 shown in stored variable)\n"
    "- Top kpi_name counts (top examples):\n"
    "  * OSLinux-OSLinux_ZABBIX_Host_Uptime: 374\n"
    "  * OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies: 374\n"
    "  * OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount: 374\n"
    "  * OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT: 372\n"
    "  * OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT: 372\n"
    "  (full top-20 kpi list is stored in metric_kpi_counts)\n\n"
    "2) trace.csv\n"
    "- Total rows: 6,408\n"
    "- Timestamp range: 2021-03-04 07:00:00 to 2021-03-04 07:29:00\n"
    "- Sample of unique cmdb_id values: IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, Tomcat... (first 50 stored)\n"
    "- Top trace_name counts (top examples):\n"
    "  * trace.self.duration_p95: 360\n"
    "  * trace.self.row_count: 360\n"
    "  * trace.self.duration_mean: 360\n"
    "  * trace.to_MG02.duration_p95: 217\n"
    "  * trace.to_MG02.row_count: 217\n"
    "  (full top-20 trace list is stored in trace_name_counts)\n\n"
    "3) log.csv\n"
    "- Total rows: 435\n"
    "- Timestamp range: 2021-03-04 07:00:00 to 2021-03-04 07:30:00\n"
    "- Sample of unique cmdb_id values: Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache... (first 50 stored)\n"
    "- Top log_name counts:\n"
    "  * log.error_count: 184\n"
    "  * log.row_count: 184\n"
    "  * log.gc_pause_max: 67\n\n"
    "4) error_logs.csv\n"
    "- Total rows: 0 (empty)\n"
    "- No timestamp range or cmdb_id entries available\n"
    "- No message counts to report (error_message_counts is empty)\n\n"
    "Stored DataFrames for follow-up: metric_df, trace_df, log_df, error_df.\n"
    "Compact summary tables are available in these variables: "
    "metric_overview, metric_kpi_counts, trace_overview, trace_name_counts, "
    "log_overview, log_name_counts, error_overview, error_message_counts, error_cmdb_summary."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    rows  min_timestamp_epoch        min_timestamp_local  max_timestamp_epoch        max_timestamp_local                             unique_cmdb_ids_sample
0  28061           1614812400  2021-03-04T07:00:00+08:00           1614814200  2021-03-04T07:30:00+08:00  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...,                                              kpi_name  count
0                  OSLinux-OSLinux_ZABBIX_Host_Uptime    374
1       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    374
2          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    374
3        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    372
4      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    372
5   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    367
6                         OSLinux-CPU_CPU_CPUidleutil    367
7                             OSLinux-CPU_CPU_CPULoad    367
8                          OSLinux-CPU_CPU_CPUSysTime    367
9                         OSLinux-CPU_CPU_CPUUserTime    367
10                             OSLinux-CPU_CPU_CPUWio    367
11                                            app.mrt    326
12                                             app.sr    326
13                                             app.rr    326
14                                            app.cnt    326
15              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    290
16           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem    290
17          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    290
18             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem    290
19       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    289,    rows  min_timestamp_epoch        min_timestamp_local  max_timestamp_epoch        max_timestamp_local                             unique_cmdb_ids_sample
0  6408           1614812400  2021-03-04T07:00:00+08:00           1614814140  2021-03-04T07:29:00+08:00  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...,                        trace_name  count
0         trace.self.duration_p95    360
1            trace.self.row_count    360
2        trace.self.duration_mean    360
3      trace.to_MG02.duration_p95    217
4         trace.to_MG02.row_count    217
5     trace.to_MG02.duration_mean    217
6      trace.to_MG01.duration_p95    212
7     trace.to_MG01.duration_mean    212
8         trace.to_MG01.row_count    212
9       trace.from_IG01.row_count    120
10      trace.from_IG02.row_count    120
11   trace.from_IG02.duration_p95    120
12  trace.from_IG02.duration_mean    120
13  trace.from_IG01.duration_mean    120
14   trace.from_IG01.duration_p95    120
15  trace.from_MG01.duration_mean    100
16   trace.from_MG01.duration_p95    100
17      trace.from_MG01.row_count    100
18   trace.from_MG02.duration_p95    100
19  trace.from_MG02.duration_mean    100,    rows  min_timestamp_epoch        min_timestamp_local  max_timestamp_epoch        max_timestamp_local                             unique_cmdb_ids_sample
0   435           1614812400  2021-03-04T07:00:00+08:00           1614814200  2021-03-04T07:30:00+08:00  [Tomcat01, Tomcat02, Tomcat03, Tomcat04, apach...,            log_name  count
0   log.error_count    184
1     log.row_count    184
2  log.gc_pause_max     67,    rows min_timestamp_epoch min_timestamp_local max_timestamp_epoch max_timestamp_local unique_cmdb_ids_sample
0     0                None                None                None                None                     [], Empty DataFrame
Columns: [message, count]
Index: [], Empty DataFrame
Columns: [cmdb_id, earliest_timestamp_local, message_count]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing metric_df from kernel
# Ensure columns and types
mdf = metric_df.copy()
mdf['timestamp'] = pd.to_numeric(mdf['timestamp'], errors='coerce').astype('Int64')
mdf['value'] = pd.to_numeric(mdf['value'], errors='coerce')
mdf['cmdb_id'] = mdf['cmdb_id'].astype(str)
mdf['kpi_name'] = mdf['kpi_name'].astype(str)

# 1) Compute global thresholds per (cmdb_id, kpi_name)
grp = mdf.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']

metric_thresholds = grp.agg(
    p95=lambda s: s.quantile(0.95) if s.count() > 0 else np.nan,
    p05=lambda s: s.quantile(0.05) if s.count() > 0 else np.nan,
    median=lambda s: s.quantile(0.5) if s.count() > 0 else np.nan,
    global_max=lambda s: s.max() if s.count() > 0 else np.nan,
    total_points=lambda s: int(s.count())
).reset_index()

# 2) Filter metric_df to incident time window (inclusive)
start_ts = 1614812400
end_ts = 1614814200
metric_incident = mdf[(mdf['timestamp'].notna()) & (mdf['timestamp'] >= start_ts) & (mdf['timestamp'] <= end_ts)].copy()

# 3) For each (cmdb_id, kpi_name) in metric_incident compute anomaly metrics
# Merge global thresholds into incident rows
mi_merged = metric_incident.merge(
    metric_thresholds[['cmdb_id', 'kpi_name', 'p95', 'p05']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Mark breaches
mi_merged['is_ge_p95'] = (mi_merged['value'] >= mi_merged['p95']).fillna(False)
mi_merged['is_le_p05'] = (mi_merged['value'] <= mi_merged['p05']).fillna(False)
mi_merged['is_breach'] = mi_merged['is_ge_p95'] | mi_merged['is_le_p05']

# Aggregate per pair within incident window
agg = mi_merged.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    incident_points=('value', lambda s: int(s.count())),
    incident_max=('value', lambda s: float(s.max()) if s.count()>0 else np.nan),
    incident_min=('value', lambda s: float(s.min()) if s.count()>0 else np.nan),
    count_ge_p95=('is_ge_p95', 'sum'),
    count_le_p05=('is_le_p05', 'sum'),
).reset_index()

# earliest_breach_ts
earliest = mi_merged[mi_merged['is_breach']].groupby(['cmdb_id','kpi_name'], dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_breach_ts'})
agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')

# Merge p95/p05 for ratio calculations
agg = agg.merge(metric_thresholds[['cmdb_id','kpi_name','p95','p05']], on=['cmdb_id','kpi_name'], how='left')

# Compute ratios with safe checks
def safe_div(numer, denom):
    if pd.isna(denom) or denom == 0:
        return np.nan
    return float(numer) / float(denom)

agg['max_over_p95_ratio'] = agg.apply(lambda r: safe_div(r['incident_max'], r['p95']), axis=1)
agg['min_over_p05_ratio'] = agg.apply(lambda r: safe_div(r['incident_min'], r['p05']), axis=1)

# Arrange columns and store result
metric_incident_anomalies = agg[[
    'cmdb_id','kpi_name','incident_points','incident_max','incident_min',
    'count_ge_p95','count_le_p05','earliest_breach_ts','p95','p05',
    'max_over_p95_ratio','min_over_p05_ratio'
]].copy()

# 4) Produce compact sorted list top 20 by total_breaches
metric_incident_anomalies['total_breaches'] = metric_incident_anomalies['count_ge_p95'] + metric_incident_anomalies['count_le_p05']
top20_summary = metric_incident_anomalies.sort_values('total_breaches', ascending=False)[
    ['cmdb_id','kpi_name','p95','p05','incident_points','count_ge_p95','count_le_p05','earliest_breach_ts','incident_max','incident_min','max_over_p95_ratio']
].head(20).reset_index(drop=True)

# Total distinct (cmdb_id, kpi_name) pairs scanned in the incident window
total_pairs_count = int(metric_incident_anomalies.shape[0])

# Keep variables stored in kernel:
# metric_thresholds, metric_incident, metric_incident_anomalies

# Display compact outputs: show first 20 thresholds, incident shape, first 20 anomalies, top20 summary, and total_pairs_count
metric_thresholds.head(20), metric_incident.shape, metric_incident_anomalies.head(20), top20_summary, total_pairs_count

```
Out[2]:
```
```python
summary = (
    "Summary of metric analysis (plain English):\n\n"
    "1) Thresholds computed:\n"
    "- Global per-(cmdb_id, kpi_name) P95, P05, median and global max were computed and stored in 'metric_thresholds'.\n\n"
    "2) Incident window:\n"
    "- Window used: 2021-03-04 07:00:00 to 2021-03-04 07:30:00 (epoch 1614812400 to 1614814200).\n"
    f"- Rows in window (metric_incident): {metric_incident.shape[0]} rows.\n\n"
    "3) Aggregated incident anomalies:\n"
    f"- Distinct (cmdb_id, kpi_name) pairs scanned in window: {total_pairs_count}.\n"
    "- Per-pair incident aggregates are stored in 'metric_incident_anomalies'.\n\n"
    "4) Top observations (from the ranked summary):\n"
    "- Several KPI pairs show breaches on every point in the window (incident_points = 31) with both count_ge_p95 and count_le_p05 equal to 31 => total_breaches = 62. Example top entries:\n"
    "  * IG01 - JVM-Memory_7778_JVM_Memory_HeapMemoryMax: incident_points=31, count_ge_p95=31, count_le_p05=31, earliest_breach_ts=1614812400, incident_max=2.040136e+09, max_over_p95_ratio=1.0\n"
    "  * IG01 - JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed: incident_points=31, count_ge_p95=31, count_le_p05=31, earliest_breach_ts=1614812400, incident_max=8.266182e+08, max_over_p95_ratio=1.0\n"
    "  * IG01 - OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT: incident_points=31, p95=0, p05=0, count_ge_p95=31, count_le_p05=31, earliest_breach_ts=1614812400 (ratios undefined when p95/p05 == 0).\n\n"
    "5) Notes and implications:\n"
    "- Many KPI thresholds (P95/P05) are exact values present in the series; when P95 or P05 equals 0, ratio calculations are undefined (NaN).\n"
    "- Some memory and CPU KPIs for IG01 show full-window breaches (consistent hits against the global thresholds).\n\n"
    "Stored variables for follow-up: metric_thresholds, metric_incident, metric_incident_anomalies (and top20 summary in 'top20_summary')."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                         kpi_name           p95           p05        median    global_max  total_points
0     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09            31
1     IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.326889e+01  9.963995e+00  2.086800e+01  3.416730e+01            20
2     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  7.070417e+08  2.230727e+08  4.467635e+08  7.341292e+08            31
3     IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.266182e+08  8.266182e+08  8.266182e+08  8.266182e+08            31
4     IG01        JVM-Operating System_7778_JVM_JVM_CPULoad  9.640000e-02  7.480000e-02  8.720000e-02  1.034000e-01            21
5     IG01                  JVM-Runtime_7778_JVM_JVM_Uptime  1.877421e+06  1.875801e+06  1.876581e+06  1.877481e+06            21
6     IG01         JVM-Threads_7778_JVM_ThreadCount_Threads  9.100000e+01  8.500000e+01  8.800000e+01  9.200000e+01            31
7     IG01                  OSLinux-CPU_CPU-0_SingleCpuUtil  9.930200e+01  6.318500e-01  7.981000e-01  1.000000e+02            11
8     IG01                  OSLinux-CPU_CPU-0_SingleCpuidle  9.949722e+01  1.047000e+00  9.920160e+01  9.958410e+01            16
9     IG01                  OSLinux-CPU_CPU-2_SingleCpuUtil  9.812000e-01  5.655500e-01  6.320000e-01  9.817000e-01            11
10    IG01                  OSLinux-CPU_CPU-2_SingleCpuidle  9.944290e+01  9.901008e+01  9.937628e+01  9.946782e+01            16
11    IG01                       OSLinux-CPU_CPU_CPUCpuUtil  2.580058e+01  2.537152e+01  2.550460e+01  2.594460e+01            20
12    IG01                          OSLinux-CPU_CPU_CPULoad  1.120000e+00  1.000000e+00  1.000000e+00  1.470000e+00            31
13    IG01                       OSLinux-CPU_CPU_CPUSysTime  4.011000e-01  1.829000e-01  2.494000e-01  4.573230e-01            31
14    IG01                      OSLinux-CPU_CPU_CPUUserTime  2.516525e+01  2.505300e+01  2.509250e+01  2.519436e+01            31
15    IG01                           OSLinux-CPU_CPU_CPUWio  3.330000e-02  8.306500e-03  1.250000e-02  2.494000e-01            31
16    IG01                      OSLinux-CPU_CPU_CPUidleutil  7.462738e+01  7.425698e+01  7.451160e+01  7.463946e+01            31
17    IG01    OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace  9.954000e+03  9.954000e+03  9.954000e+03  9.954000e+03            11
18    IG01          OSLinux-OSLinux_FILESYSTEM_-_FSCapacity  3.024600e+01  3.024590e+01  3.024595e+01  3.024600e+01            16
19    IG01  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent  1.636630e+01  1.636627e+01  1.636630e+01  1.636630e+01            16, (28061, 4),    cmdb_id                                         kpi_name  incident_points  incident_max  incident_min  count_ge_p95  count_le_p05  earliest_breach_ts           p95           p05  max_over_p95_ratio  min_over_p05_ratio  total_breaches
0     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31  2.040136e+09  2.040136e+09            31            31          1614812400  2.040136e+09  2.040136e+09            1.000000            1.000000              62
1     IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage               20  3.416730e+01  9.308400e+00             1             1          1614812400  3.326889e+01  9.963995e+00            1.027005            0.934204               2
2     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed               31  7.341292e+08  2.039828e+08             2             2          1614812400  7.070417e+08  2.230727e+08            1.038311            0.914423               4
3     IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               31  8.266182e+08  8.266182e+08            31            31          1614812400  8.266182e+08  8.266182e+08            1.000000            1.000000              62
4     IG01        JVM-Operating System_7778_JVM_JVM_CPULoad               21  1.034000e-01  7.070000e-02             2             2          1614812460  9.640000e-02  7.480000e-02            1.072614            0.945187               4
5     IG01                  JVM-Runtime_7778_JVM_JVM_Uptime               21  1.877481e+06  1.875681e+06             2             2          1614812400  1.877421e+06  1.875801e+06            1.000032            0.999936               4
6     IG01         JVM-Threads_7778_JVM_ThreadCount_Threads               31  9.200000e+01  8.500000e+01             2            10          1614812580  9.100000e+01  8.500000e+01            1.010989            1.000000              12
7     IG01                  OSLinux-CPU_CPU-0_SingleCpuUtil               11  1.000000e+02  5.987000e-01             1             1          1614812760  9.930200e+01  6.318500e-01            1.007029            0.947535               2
8     IG01                  OSLinux-CPU_CPU-0_SingleCpuidle               16  9.958410e+01  0.000000e+00             1             1          1614812640  9.949722e+01  1.047000e+00            1.000873            0.000000               2
9     IG01                  OSLinux-CPU_CPU-2_SingleCpuUtil               11  9.817000e-01  5.654000e-01             1             1          1614812880  9.812000e-01  5.655500e-01            1.000510            0.999735               2
10    IG01                  OSLinux-CPU_CPU-2_SingleCpuidle               16  9.946782e+01  9.898540e+01             1             1          1614812520  9.944290e+01  9.901008e+01            1.000251            0.999751               2
11    IG01                       OSLinux-CPU_CPU_CPUCpuUtil               20  2.594460e+01  2.536050e+01             1             1          1614813060  2.580058e+01  2.537152e+01            1.005582            0.999566               2
12    IG01                          OSLinux-CPU_CPU_CPULoad               31  1.470000e+00  1.000000e+00             2            24          1614812400  1.120000e+00  1.000000e+00            1.312500            1.000000              26
13    IG01                       OSLinux-CPU_CPU_CPUSysTime               31  4.573230e-01  1.704000e-01             2             3          1614813000  4.011000e-01  1.829000e-01            1.140172            0.931657               5
14    IG01                      OSLinux-CPU_CPU_CPUUserTime               31  2.519436e+01  2.505200e+01             2             2          1614813000  2.516525e+01  2.505300e+01            1.001157            0.999960               4
15    IG01                           OSLinux-CPU_CPU_CPUWio               31  2.494000e-01  8.300000e-03             3             2          1614812760  3.330000e-02  8.306500e-03            7.489489            0.999217               5
16    IG01                      OSLinux-CPU_CPU_CPUidleutil               31  7.463946e+01  7.405540e+01             2             2          1614813000  7.462738e+01  7.425698e+01            1.000162            0.997285               4
17    IG01    OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace               11  9.954000e+03  9.954000e+03            11            11          1614812520  9.954000e+03  9.954000e+03            1.000000            1.000000              22
18    IG01          OSLinux-OSLinux_FILESYSTEM_-_FSCapacity               16  3.024600e+01  3.024590e+01             4             7          1614812400  3.024600e+01  3.024590e+01            1.000000            1.000000              11
19    IG01  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent               16  1.636630e+01  1.636627e+01            11             5          1614812400  1.636630e+01  1.636627e+01            1.000000            1.000000              16,          cmdb_id                                           kpi_name           p95           p05  incident_points  count_ge_p95  count_le_p05  earliest_breach_ts  incident_max  incident_min  max_over_p95_ratio
0           IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
1           IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.266182e+08  8.266182e+08               31            31            31          1614812400  8.266182e+08  8.266182e+08                 1.0
2       apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
3       apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
4       apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
5       apache02   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount  5.000000e+00  5.000000e+00               31            31            31          1614812400  5.000000e+00  5.000000e+00                 1.0
6       apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00  7.000000e+00               31            31            31          1614812400  7.000000e+00  7.000000e+00                 1.0
7           IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09               31            31            31          1614812400  2.040136e+09  2.040136e+09                 1.0
8   ServiceTest7                                             app.rr  1.000000e+02  1.000000e+02               31            31            31          1614812400  1.000000e+02  1.000000e+02                 1.0
9           IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00  1.000000e+00               31            31            31          1614812400  1.000000e+00  1.000000e+00                 1.0
10      apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  4.000000e+00  4.000000e+00               31            31            31          1614812400  4.000000e+00  4.000000e+00                 1.0
11       Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  4.000000e+00  4.000000e+00               31            31            31          1614812400  4.000000e+00  4.000000e+00                 1.0
12       Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
13       Redis02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
14       Redis02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
15       Redis02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01               31            31            31          1614812400  3.000000e-01  3.000000e-01                 1.0
16      Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02  1.000000e+02               31            31            31          1614812400  1.000000e+02  1.000000e+02                 1.0
17      Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
18       Redis02    redis-Redis_6379_Redis  (used_cpu_sys_children)  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN
19       Redis02   redis-Redis_6379_Redis  (used_cpu_user_children)  0.000000e+00  0.000000e+00               31            31            31          1614812400  0.000000e+00  0.000000e+00                 NaN, 1685)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing variables from kernel: metric_incident, metric_thresholds
# Allowed components list
allowed_components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                      'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Prepare incident data merged with global thresholds
mi = metric_incident.copy()
# Ensure types
mi['timestamp'] = pd.to_numeric(mi['timestamp'], errors='coerce').astype('Int64')
mi['value'] = pd.to_numeric(mi['value'], errors='coerce')
mi['cmdb_id'] = mi['cmdb_id'].astype(str)
mi['kpi_name'] = mi['kpi_name'].astype(str)

mt = metric_thresholds.copy()
mt['cmdb_id'] = mt['cmdb_id'].astype(str)
mt['kpi_name'] = mt['kpi_name'].astype(str)

# Merge thresholds into incident rows
mi = mi.merge(mt[['cmdb_id','kpi_name','p95','p05']], on=['cmdb_id','kpi_name'], how='left')

# Filter to allowed components
mi_allowed = mi[mi['cmdb_id'].isin(allowed_components)].copy()

# Mark breach points (use thresholds from metric_thresholds; if missing, treat as non-breach)
mi_allowed['is_ge_p95'] = (mi_allowed['p95'].notna()) & (mi_allowed['value'] >= mi_allowed['p95'])
mi_allowed['is_le_p05'] = (mi_allowed['p05'].notna()) & (mi_allowed['value'] <= mi_allowed['p05'])
mi_allowed['is_breach'] = mi_allowed['is_ge_p95'] | mi_allowed['is_le_p05']

# We'll detect consecutive breached minutes (consecutive by timestamp difference == 60 seconds)
segments_list = []

group_cols = ['cmdb_id','kpi_name']
for (cmdb, kpi), g in mi_allowed.groupby(group_cols, sort=False):
    g = g.sort_values('timestamp').reset_index(drop=True)
    # Extract only breached rows (non-breach can't be part of consecutive breach segments)
    gb = g[g['is_breach']].copy()
    if gb.empty:
        continue
    # Compute consecutive runs by checking timestamp diff == 60
    gb['ts_diff'] = gb['timestamp'].diff().fillna(9999999).astype('Int64')
    # Start a new segment when diff != 60
    gb['segment_idx'] = (gb['ts_diff'] != 60).cumsum()
    # Aggregate each segment
    seg_agg = gb.groupby('segment_idx', sort=True).agg(
        cmdb_id=('cmdb_id','first'),
        kpi_name=('kpi_name','first'),
        start_ts=('timestamp','min'),
        end_ts=('timestamp','max'),
        length_minutes=('timestamp','count'),
        segment_max_value=('value','max'),
        segment_min_value=('value','min'),
        p95=('p95','first'),
        p05=('p05','first')
    ).reset_index(drop=True)
    # Keep only segments length >=3
    seg_agg = seg_agg[seg_agg['length_minutes'] >= 3].copy()
    if seg_agg.empty:
        continue
    # Compute ratios safely
    def safe_ratio(num, den):
        if pd.isna(den) or den == 0:
            return np.nan
        return float(num) / float(den)
    seg_agg['max_over_p95_ratio'] = seg_agg.apply(lambda r: safe_ratio(r['segment_max_value'], r['p95']), axis=1)
    seg_agg['min_over_p05_ratio'] = seg_agg.apply(lambda r: safe_ratio(r['segment_min_value'], r['p05']), axis=1)
    segments_list.append(seg_agg)

# Concatenate all detected segments
if segments_list:
    metric_fault_segments = pd.concat(segments_list, ignore_index=True)
else:
    metric_fault_segments = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','start_ts','end_ts','length_minutes',
        'segment_max_value','segment_min_value','p95','p05',
        'max_over_p95_ratio','min_over_p05_ratio'
    ])

# Store metric_fault_segments in kernel (as required)
# 3) Aggregate faults per component
if not metric_fault_segments.empty:
    component_fault_summary = metric_fault_segments.groupby('cmdb_id', sort=True).agg(
        faults_count=('kpi_name','count'),
        earliest_fault_start=('start_ts','min')
    ).reset_index().sort_values('faults_count', ascending=False).reset_index(drop=True)
else:
    component_fault_summary = pd.DataFrame(columns=['cmdb_id','faults_count','earliest_fault_start'])

# 4) Prepare compact top-20 segments sorted by severity = max_over_p95_ratio (treat NaN as 0)
if not metric_fault_segments.empty:
    display_df = metric_fault_segments.copy()
    display_df['severity'] = display_df['max_over_p95_ratio'].fillna(0)
    top20_segments = display_df.sort_values('severity', ascending=False)[
        ['cmdb_id','kpi_name','start_ts','end_ts','length_minutes','p95','p05','segment_max_value','max_over_p95_ratio']
    ].head(20).reset_index(drop=True)
else:
    top20_segments = metric_fault_segments.head(0)[
        ['cmdb_id','kpi_name','start_ts','end_ts','length_minutes','p95','p05','segment_max_value','max_over_p95_ratio']
    ]

total_detected_segments = int(metric_fault_segments.shape[0])

# Keep outputs compact for the controller
metric_fault_segments.shape, total_detected_segments, top20_segments, component_fault_summary.head(20)

```
Out[3]:
```
```python
# Plain-English summary of detected metric fault segments (compact)
# Uses existing kernel variables: metric_fault_segments, total_detected_segments, top20_segments, component_fault_summary

# Build a concise textual summary
td = total_detected_segments if 'total_detected_segments' in globals() else (metric_fault_segments.shape[0] if 'metric_fault_segments' in globals() else 0)
top_rows = top20_segments.head(5) if 'top20_segments' in globals() else None
comp_top = component_fault_summary.head(10) if 'component_fault_summary' in globals() else None

lines = []
lines.append(f"Detected metric fault segments: {td} segments (stored in 'metric_fault_segments').")
lines.append("Kept only segments with >=3 consecutive anomalous minutes (breach = value >= p95 or <= p05).")
lines.append("")
lines.append("Top severity segments (top 5 by max_over_p95_ratio):")
if top_rows is None or top_rows.empty:
    lines.append("  (no segments detected)")
else:
    for i, r in top_rows.reset_index(drop=True).iterrows():
        lines.append(
            f"  {i+1}. {r['cmdb_id']} | {r['kpi_name']} | start={int(r['start_ts'])} end={int(r['end_ts'])} "
            f"len={int(r['length_minutes'])}min | p95={r['p95']:.6g} p05={r['p05']:.6g} | "
            f"max={r['segment_max_value']:.6g} | max_over_p95_ratio={(r['max_over_p95_ratio'] if pd.notna(r['max_over_p95_ratio']) else 'NaN')}"
        )
lines.append("")
lines.append("Per-component fault summary (top components by faults_count):")
if comp_top is None or comp_top.empty:
    lines.append("  (no component faults)")
else:
    for i, r in comp_top.reset_index(drop=True).iterrows():
        lines.append(f"  {i+1}. {r['cmdb_id']} — faults_count={int(r['faults_count'])}, earliest_fault_start={int(r['earliest_fault_start'])}")

lines.append("")
lines.append("Notes:")
lines.append("  - Ratios are segment_max_value / p95 (NaN when p95 is 0 or missing).")
lines.append("  - Result tables stored for further analysis: 'metric_fault_segments' and 'component_fault_summary'.")
summary = "\n".join(lines)

# Display the concise summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((222, 11), 222,      cmdb_id                                           kpi_name    start_ts      end_ts  length_minutes           p95           p05  segment_max_value  max_over_p95_ratio
0    Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1614813120  1614814200              19  1.000000e+00  1.000000e+00       2.000000e+00            2.000000
1   apache01                            OSLinux-CPU_CPU_CPULoad  1614812700  1614813240              10  6.000000e-02  0.000000e+00       1.000000e-01            1.666667
2    Mysql02                            OSLinux-CPU_CPU_CPULoad  1614812760  1614812940               4  3.950000e-01  9.000000e-02       6.300000e-01            1.594937
3       IG01                            OSLinux-CPU_CPU_CPULoad  1614813360  1614814200              15  1.120000e+00  1.000000e+00       1.470000e+00            1.312500
4    Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1614812400  1614814200              31  4.500000e-01  4.000000e-01       5.000000e-01            1.111111
5   apache01                             OSLinux-CPU_CPU_CPUWio  1614813240  1614813420               4  2.285350e-02  8.300000e-03       2.490700e-02            1.089855
6       IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  1614812700  1614812880               4  1.037701e+09  2.556274e+08       1.069526e+09            1.030669
7       IG02                            OSLinux-CPU_CPU_CPULoad  1614812520  1614812640               3  1.060000e+00  1.000000e+00       1.070000e+00            1.009434
8       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  1614812400  1614814200              31  3.000000e+00  3.000000e+00       3.000000e+00            1.000000
9       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1614812400  1614814200              31  2.040136e+09  2.040136e+09       2.040136e+09            1.000000
10      IG02                            OSLinux-CPU_CPU_CPULoad  1614813600  1614814080               9  1.060000e+00  1.000000e+00       1.060000e+00            1.000000
11      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  1614812400  1614814200              31  8.266182e+08  8.266182e+08       8.266182e+08            1.000000
12      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1614812400  1614814200              31  1.000000e+00  1.000000e+00       1.000000e+00            1.000000
13      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  1614812400  1614814200              31  2.040136e+09  2.040136e+09       2.040136e+09            1.000000
14      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  1614812400  1614814200              31  2.040136e+09  2.040136e+09       2.040136e+09            1.000000
15      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  1614812400  1614814200              31  5.060958e+08  5.060958e+08       5.060958e+08            1.000000
16      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  1614812400  1614814200              31  8.301933e+08  8.301933e+08       8.301933e+08            1.000000
17      IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1614812400  1614814200              31  1.000000e+00  1.000000e+00       1.000000e+00            1.000000
18      IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1614812400  1614814200              31  4.000000e-01  3.000000e-01       4.000000e-01            1.000000
19      IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  1614812400  1614814200              31  3.000000e+00  3.000000e+00       3.000000e+00            1.000000,      cmdb_id  faults_count  earliest_fault_start
0    Redis01            33            1614812400
1    Redis02            30            1614812400
2   Tomcat03            23            1614812400
3   Tomcat02            21            1614812400
4   Tomcat04            21            1614812400
5   Tomcat01            19            1614812400
6       IG02            14            1614812400
7       IG01            12            1614812400
8   apache01            12            1614812400
9       MG02            12            1614812400
10  apache02             9            1614812400
11   Mysql01             8            1614812400
12   Mysql02             6            1614812400
13      MG01             2            1614812400)```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing trace_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Prepare trace dataframe
tdf = trace_df.copy()
tdf['timestamp'] = pd.to_numeric(tdf['timestamp'], errors='coerce').astype('Int64')
tdf['value'] = pd.to_numeric(tdf['value'], errors='coerce')
tdf['cmdb_id'] = tdf['cmdb_id'].astype(str)
tdf['trace_name'] = tdf['trace_name'].astype(str)

# 1) Global thresholds per (cmdb_id, trace_name)
trace_thresholds = (
    tdf.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        p95=lambda s: s.quantile(0.95) if s.count()>0 else np.nan,
        p05=lambda s: s.quantile(0.05) if s.count()>0 else np.nan,
        median=lambda s: s.quantile(0.5) if s.count()>0 else np.nan,
        total_points=lambda s: int(s.count())
    )
    .reset_index()
)

# 2) Filter trace_df to incident window
start_ts = 1614812400
end_ts = 1614814200
trace_incident = tdf[(tdf['timestamp'].notna()) & (tdf['timestamp'] >= start_ts) & (tdf['timestamp'] <= end_ts)].copy()

# 3) Per-(cmdb_id, trace_name) incident aggregations using global thresholds
# Merge thresholds
ti = trace_incident.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Mark breaches
ti['is_ge_p95'] = ti['p95'].notna() & (ti['value'] >= ti['p95'])
ti['is_le_p05'] = ti['p05'].notna() & (ti['value'] <= ti['p05'])
ti['is_breach'] = ti['is_ge_p95'] | ti['is_le_p05']

# Aggregate per pair
agg = ti.groupby(['cmdb_id','trace_name'], dropna=False).agg(
    incident_points=('value','count'),
    incident_max=('value', lambda s: float(s.max()) if s.count()>0 else np.nan),
    incident_min=('value', lambda s: float(s.min()) if s.count()>0 else np.nan),
    count_ge_p95=('is_ge_p95','sum'),
    count_le_p05=('is_le_p05','sum')
).reset_index()

# earliest_breach_ts
earliest = ti[ti['is_breach']].groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_breach_ts'})
trace_incident_anomalies = agg.merge(trace_thresholds[['cmdb_id','trace_name','p95','p05']], on=['cmdb_id','trace_name'], how='left')
trace_incident_anomalies = trace_incident_anomalies.merge(earliest, on=['cmdb_id','trace_name'], how='left')

# severity = incident_max / p95 if p95 > 0 else NaN
def compute_severity(row):
    p95 = row['p95']
    imax = row['incident_max']
    if pd.isna(p95) or p95 == 0:
        return np.nan
    return float(imax) / float(p95)

trace_incident_anomalies['severity'] = trace_incident_anomalies.apply(compute_severity, axis=1)

# total_breaches for sorting
trace_incident_anomalies['total_breaches'] = trace_incident_anomalies['count_ge_p95'] + trace_incident_anomalies['count_le_p05']

# 4) Top-20 list sorted by total_breaches desc then severity desc
top20_trace_anomalies = (
    trace_incident_anomalies
    .sort_values(['total_breaches','severity'], ascending=[False, False])
    .loc[:, ['cmdb_id','trace_name','p95','p05','incident_points','count_ge_p95','count_le_p05','earliest_breach_ts','incident_max','severity']]
    .head(20)
    .reset_index(drop=True)
)

# Per-component aggregate: count of trace anomaly pairs (where total_breaches>0) and earliest breach ts per component
anom_pairs = trace_incident_anomalies[trace_incident_anomalies['total_breaches'] > 0].copy()
trace_component_summary = (
    anom_pairs.groupby('cmdb_id', sort=True).agg(
        total_trace_anomaly_pairs=('trace_name','count'),
        earliest_trace_breach_ts=('earliest_breach_ts','min')
    ).reset_index().sort_values('total_trace_anomaly_pairs', ascending=False).head(20).reset_index(drop=True)
)

# Keep full tables in kernel: trace_thresholds, trace_incident, trace_incident_anomalies, trace_component_summary, top20_trace_anomalies
# Return compact summaries: top20_trace_anomalies and trace_component_summary
top20_trace_anomalies, trace_component_summary

```
Out[4]:
```
```python
# Compact plain-English summary of trace anomaly analysis
summary = (
    "Trace anomaly analysis summary (using incident window 2021-03-04 07:00:00 to 07:30:00 UTC+8):\n\n"
    "What was computed and stored:\n"
    "- trace_thresholds: global p95, p05, median, total_points per (cmdb_id, trace_name).\n"
    "- trace_incident: trace rows filtered to the incident window.\n"
    "- trace_incident_anomalies: per-pair incident aggregates (counts, breaches, earliest breach, severity).\n\n"
    "Top anomalous trace pairs (compact top-20 by total_breaches):\n"
    "1) dockerB1 | trace.self.duration_p95 — p95=0.011, p05=0.010, incident_points=30, count_ge_p95=15, count_le_p05=10, earliest_breach_ts=1614812400, incident_max=0.01135, severity≈1.0318\n"
    "2) dockerA2 | trace.self.duration_p95 — p95=0.471338, p05=0.001, incident_points=30, count_ge_p95=2, count_le_p05=22, earliest_breach_ts=1614812460, incident_max=0.47730, severity≈1.0127\n"
    "3) dockerA1 | trace.self.duration_p95 — p95=0.454705, p05=0.002, incident_points=30, count_ge_p95=2, count_le_p05=13, earliest_breach_ts=1614812520, incident_max=0.48535, severity≈1.0674\n"
    "4) MG01 | trace.from_dockerA2.row_count — p95=6.85, p05=1, incident_points=24, count_ge_p95=2, count_le_p05=8, earliest_breach_ts=1614812700, incident_max=9.0, severity≈1.3139\n"
    "5) dockerA2 | trace.to_MG01.row_count — same stats as #4 (pair across services)\n"
    "(Other top entries include MG02/dockers, IG01/Tomcat links and more; full top-20 stored in 'top20_trace_anomalies')\n\n"
    "Per-component aggregate (top components by number of trace-anomaly pairs):\n"
    "- MG02: 42 anomalous trace pairs, earliest breach at epoch 1614812400\n"
    "- MG01: 42 anomalous trace pairs, earliest breach at epoch 1614812400\n"
    "- Tomcat03: 18 pairs (earliest 1614812400)\n"
    "- Tomcat01, dockerB1, dockerB2, dockerA2, dockerA1: each 18 pairs (earliest 1614812400)\n"
    "- IG01: 15 pairs (earliest 1614812400); IG02: 15 pairs (earliest 1614812460)\n\n"
    "Notes and next steps:\n"
    "- 'severity' is incident_max / p95 (NaN if p95 is zero). Values above 1 indicate incident_max exceeded the global p95.\n"
    "- Earliest breach timestamps are provided as epoch seconds (UTC+8 context used earlier).\n"
    "- Full tables available in the kernel for follow-up: trace_thresholds, trace_incident, trace_incident_anomalies, trace_component_summary, top20_trace_anomalies.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                        trace_name        p95    p05  incident_points  count_ge_p95  count_le_p05  earliest_breach_ts  incident_max  severity
0   dockerB1           trace.self.duration_p95   0.011000  0.010               30            15            10          1614812400       0.01135  1.031818
1   dockerA2           trace.self.duration_p95   0.471338  0.001               30             2            22          1614812460       0.47730  1.012650
2   dockerA1           trace.self.duration_p95   0.454705  0.002               30             2            13          1614812520       0.48535  1.067395
3       MG01     trace.from_dockerA2.row_count   6.850000  1.000               24             2             8          1614812700       9.00000  1.313869
4   dockerA2           trace.to_MG01.row_count   6.850000  1.000               24             2             8          1614812700       9.00000  1.313869
5       MG01     trace.from_dockerA1.row_count   6.150000  1.000               18             1             6          1614812760       7.00000  1.138211
6   dockerA1           trace.to_MG01.row_count   6.150000  1.000               18             1             6          1614812760       7.00000  1.138211
7       MG02       trace.to_dockerA2.row_count  18.000000  6.000               25             3             4          1614812400      20.00000  1.111111
8   dockerA2         trace.from_MG02.row_count  18.000000  6.000               25             3             4          1614812400      20.00000  1.111111
9       MG02     trace.from_dockerA1.row_count   6.550000  1.000               24             2             5          1614812880       7.00000  1.068702
10  dockerA1           trace.to_MG02.row_count   6.550000  1.000               24             2             5          1614812880       7.00000  1.068702
11      MG02     trace.from_dockerB1.row_count  16.000000  5.200               25             5             2          1614812760      17.00000  1.062500
12  dockerB1           trace.to_MG02.row_count  16.000000  5.200               25             5             2          1614812760      17.00000  1.062500
13      MG02  trace.from_dockerB2.duration_p95   0.105350  0.007               25             2             4          1614812700       0.24345  2.310869
14  dockerB2        trace.to_MG02.duration_p95   0.105350  0.007               25             2             4          1614812700       0.24345  2.310869
15      IG01       trace.to_Tomcat01.row_count  17.000000  6.000               30             3             3          1614812520      21.00000  1.235294
16  Tomcat01         trace.from_IG01.row_count  17.000000  6.000               30             3             3          1614812520      21.00000  1.235294
17      IG02       trace.to_Tomcat02.row_count  18.100000  7.000               30             2             4          1614812580      22.00000  1.215470
18  Tomcat02         trace.from_IG02.row_count  18.100000  7.000               30             2             4          1614812580      22.00000  1.215470
19      IG01       trace.to_Tomcat02.row_count  17.000000  5.450               30             4             2          1614812400      20.00000  1.176471,      cmdb_id  total_trace_anomaly_pairs  earliest_trace_breach_ts
0       MG02                         42                1614812400
1       MG01                         42                1614812400
2   Tomcat03                         18                1614812400
3   Tomcat01                         18                1614812400
4   dockerB1                         18                1614812400
5   dockerB2                         18                1614812400
6   dockerA2                         18                1614812400
7   dockerA1                         18                1614812400
8       IG01                         15                1614812400
9       IG02                         15                1614812460
10  Tomcat04                         15                1614812400
11  Tomcat02                         15                1614812400)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd

# Reuse existing log_df from kernel
ldf = log_df.copy()
ldf['timestamp'] = pd.to_numeric(ldf['timestamp'], errors='coerce').astype('Int64')
ldf['value'] = pd.to_numeric(ldf['value'], errors='coerce')
ldf['cmdb_id'] = ldf['cmdb_id'].astype(str)
ldf['log_name'] = ldf['log_name'].astype(str)

# 1) Global thresholds per (cmdb_id, log_name)
log_thresholds = (
    ldf.groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(
        p95=lambda s: s.quantile(0.95) if s.count() > 0 else np.nan,
        p05=lambda s: s.quantile(0.05) if s.count() > 0 else np.nan,
        median=lambda s: s.quantile(0.5) if s.count() > 0 else np.nan,
        total_points=lambda s: int(s.count())
    )
    .reset_index()
)

# 2) Filter to incident window (inclusive)
start_ts = 1614812400
end_ts = 1614814200
log_incident = ldf[(ldf['timestamp'].notna()) & (ldf['timestamp'] >= start_ts) & (ldf['timestamp'] <= end_ts)].copy()

# 3) Compute incident anomalies using global thresholds (merge thresholds first)
li = log_incident.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')

li['is_ge_p95'] = li['p95'].notna() & (li['value'] >= li['p95'])
li['is_le_p05'] = li['p05'].notna() & (li['value'] <= li['p05'])
li['is_breach'] = li['is_ge_p95'] | li['is_le_p05']

# Aggregate per pair in incident window
agg = li.groupby(['cmdb_id','log_name'], dropna=False).agg(
    incident_points=('value','count'),
    incident_max=('value', lambda s: float(s.max()) if s.count()>0 else np.nan),
    incident_min=('value', lambda s: float(s.min()) if s.count()>0 else np.nan),
    count_ge_p95=('is_ge_p95','sum'),
    count_le_p05=('is_le_p05','sum')
).reset_index()

# earliest breach timestamp per pair (epoch)
earliest = li[li['is_breach']].groupby(['cmdb_id','log_name'], dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_breach_ts'})

log_incident_anomalies = agg.merge(log_thresholds[['cmdb_id','log_name','p95','p05']], on=['cmdb_id','log_name'], how='left')
log_incident_anomalies = log_incident_anomalies.merge(earliest, on=['cmdb_id','log_name'], how='left')

# severity = incident_max / p95 if p95>0 else NaN
def compute_sev(row):
    p95 = row['p95']
    imax = row['incident_max']
    if pd.isna(p95) or p95 == 0:
        return np.nan
    return float(imax) / float(p95)

log_incident_anomalies['severity'] = log_incident_anomalies.apply(compute_sev, axis=1)
log_incident_anomalies['total_breaches'] = log_incident_anomalies['count_ge_p95'] + log_incident_anomalies['count_le_p05']

# 4) Top-20 table sorted by total_breaches desc then severity desc
top20_logs = (
    log_incident_anomalies
    .sort_values(['total_breaches','severity'], ascending=[False, False])
    .loc[:, ['cmdb_id','log_name','p95','p05','incident_points','count_ge_p95','count_le_p05','earliest_breach_ts','incident_max','severity']]
    .head(20)
    .reset_index(drop=True)
)

# 5) Focused summary for Tomcat and apache components
focus_components = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','apache01','apache02']
tomcat_focus = log_incident_anomalies[log_incident_anomalies['cmdb_id'].isin(focus_components)].copy()

# Ensure we have total_breaches for sorting; it exists in log_incident_anomalies
tomcat_focus = tomcat_focus.sort_values(['total_breaches','severity'], ascending=[False, False])

# Select same output columns as top20_logs
tomcat_focus = tomcat_focus.loc[:, ['cmdb_id','log_name','p95','p05','incident_points','count_ge_p95','count_le_p05','earliest_breach_ts','incident_max','severity']].reset_index(drop=True).head(10)

# Keep full tables in kernel: log_thresholds, log_incident, log_incident_anomalies
# Return compact outputs only
top20_logs, tomcat_focus, log_thresholds.shape, log_incident.shape, log_incident_anomalies.shape

```
Out[5]:
```
```python
# Compact plain-English summary of log anomaly analysis (stored tables: log_thresholds, log_incident, log_incident_anomalies)
def fmt(x):
    if pd.isna(x):
        return "NaN"
    if isinstance(x, (int,)) and not isinstance(x, bool):
        return str(int(x))
    try:
        # show up to 6 significant digits
        return f"{float(x):.6g}"
    except:
        return str(x)

lines = []
lines.append("Log anomaly analysis summary (incident window 2021-03-04 07:00–07:30 UTC+8):")
lines.append("")
# counts
lt_rows = log_thresholds.shape[0] if 'log_thresholds' in globals() else 0
li_rows = log_incident.shape[0] if 'log_incident' in globals() else 0
la_rows = log_incident_anomalies.shape[0] if 'log_incident_anomalies' in globals() else 0
lines.append(f"- Computed log_thresholds for {lt_rows} (cmdb_id, log_name) pairs.")
lines.append(f"- Rows in incident window (log_incident): {li_rows}")
lines.append(f"- Per-pair incident anomaly summaries computed (log_incident_anomalies): {la_rows} pairs")
lines.append("")

# Top anomalies (compact): use top20_logs variable
lines.append("Top anomalous log pairs (top entries by total_breaches then severity):")
if 'top20_logs' in globals() and not top20_logs.empty:
    for i, r in top20_logs.head(8).iterrows():
        lines.append(
            f" {i+1}. {r['cmdb_id']} | {r['log_name']} — p95={fmt(r['p95'])}, p05={fmt(r['p05'])}, "
            f"points={fmt(r['incident_points'])}, ge_p95={fmt(r['count_ge_p95'])}, le_p05={fmt(r['count_le_p05'])}, "
            f"earliest={fmt(r['earliest_breach_ts'])}, max={fmt(r['incident_max'])}, severity={fmt(r['severity'])}"
        )
else:
    lines.append(" (no anomalies detected)")

lines.append("")
# Focused Tomcat & apache summary
lines.append("Focused Tomcat/apache log anomalies (up to 10 rows):")
if 'tomcat_focus' in globals() and not tomcat_focus.empty:
    for i, r in tomcat_focus.iterrows():
        lines.append(
            f" {i+1}. {r['cmdb_id']} | {r['log_name']} — p95={fmt(r['p95'])}, p05={fmt(r['p05'])}, "
            f"points={fmt(r['incident_points'])}, ge_p95={fmt(r['count_ge_p95'])}, le_p05={fmt(r['count_le_p05'])}, "
            f"earliest={fmt(r['earliest_breach_ts'])}, max={fmt(r['incident_max'])}, severity={fmt(r['severity'])}"
        )
else:
    lines.append(" (no Tomcat/apache anomalies in window)")

lines.append("")
lines.append("Notes:")
lines.append("- Severity = incident_max / p95 (NaN when p95 is 0 or missing).")
lines.append("- Many components show log.error_count with p95=0 and both count_ge_p95 and count_le_p05 equal to full-window counts (ratios undefined).")
lines.append("- Full tables stored in kernel variables for follow-up: log_thresholds, log_incident, log_incident_anomalies, top20_logs, tomcat_focus.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name       p95      p05  incident_points  count_ge_p95  count_le_p05  earliest_breach_ts  incident_max  severity
0   Tomcat01   log.error_count    0.0000   0.0000               31            31            31          1614812400          0.00       NaN
1   Tomcat02   log.error_count    0.0000   0.0000               31            31            31          1614812400          0.00       NaN
2   apache01   log.error_count    0.0000   0.0000               31            31            31          1614812400          0.00       NaN
3   apache02   log.error_count    0.0000   0.0000               31            31            31          1614812400          0.00       NaN
4   Tomcat03   log.error_count    0.0000   0.0000               30            30            30          1614812400          0.00       NaN
5   Tomcat04   log.error_count    0.0000   0.0000               30            30            30          1614812400          0.00       NaN
6   apache02     log.row_count   61.5000  33.0000               31             2             4          1614812520         64.00  1.040650
7   Tomcat01  log.gc_pause_max    5.1400   5.0750               31             4             2          1614812640          5.14  1.000000
8   Tomcat04     log.row_count   33.2000  13.0000               30             2             3          1614812460         37.00  1.114458
9   Tomcat02     log.row_count  105.0000  80.0000               31             3             2          1614812580        106.00  1.009524
10  Tomcat03     log.row_count   33.2000  13.4500               30             2             2          1614812400         37.00  1.114458
11  apache01     log.row_count   60.5000  23.5000               31             2             2          1614812520         65.00  1.074380
12  Tomcat01     log.row_count  103.0000  79.5000               31             2             2          1614812760        105.00  1.019417
13  Tomcat02  log.gc_pause_max    5.1350   5.0800               31             2             2          1614813540          5.14  1.000974
14  Tomcat04  log.gc_pause_max    0.1060   0.0700                3             1             2          1614812640          0.11  1.037736
15  Tomcat03  log.gc_pause_max    0.0695   0.0605                2             1             1          1614813120          0.07  1.007194,     cmdb_id          log_name     p95     p05  incident_points  count_ge_p95  count_le_p05  earliest_breach_ts  incident_max  severity
0  Tomcat01   log.error_count    0.00   0.000               31            31            31          1614812400          0.00       NaN
1  Tomcat02   log.error_count    0.00   0.000               31            31            31          1614812400          0.00       NaN
2  apache01   log.error_count    0.00   0.000               31            31            31          1614812400          0.00       NaN
3  apache02   log.error_count    0.00   0.000               31            31            31          1614812400          0.00       NaN
4  Tomcat03   log.error_count    0.00   0.000               30            30            30          1614812400          0.00       NaN
5  Tomcat04   log.error_count    0.00   0.000               30            30            30          1614812400          0.00       NaN
6  apache02     log.row_count   61.50  33.000               31             2             4          1614812520         64.00  1.040650
7  Tomcat01  log.gc_pause_max    5.14   5.075               31             4             2          1614812640          5.14  1.000000
8  Tomcat04     log.row_count   33.20  13.000               30             2             3          1614812460         37.00  1.114458
9  Tomcat02     log.row_count  105.00  80.000               31             3             2          1614812580        106.00  1.009524, (16, 6), (435, 4), (16, 12))```
```

In [ ]:
# In[6]:

import re
import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables: trace_incident_anomalies, component_fault_summary
tia = trace_incident_anomalies.copy()
cfs = component_fault_summary.copy()

# Allowed root-cause components
allowed_components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                      'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# 1) Parse trace_name to extract referenced component X for patterns trace.to_<X>.* and trace.from_<X>.*
# Create columns: direction ('to' or 'from'), referenced_component (X)
def extract_ref_comp(trace_name):
    # regex to capture to/from and component name (allow letters, numbers, underscores, hyphens)
    m = re.match(r'^trace\.(to|from)_([A-Za-z0-9_\-]+)', str(trace_name))
    if m:
        return m.group(1), m.group(2)
    return None, None

extracted = tia['trace_name'].apply(lambda tn: pd.Series(extract_ref_comp(tn), index=['direction','referenced_component']))
tia = pd.concat([tia.reset_index(drop=True), extracted], axis=1)

# 1) For each referenced component X, count how many trace-anomaly rows point to X and collect earliest_breach_ts
ref_stats = (
    tia[tia['referenced_component'].notna()]
    .groupby('referenced_component', sort=True)
    .agg(
        trace_anomaly_edge_count=('trace_name','count'),
        earliest_trace_breach_ts=('earliest_breach_ts', lambda s: int(s.dropna().min()) if s.dropna().size>0 else pd.NA)
    )
    .reset_index()
)

# 2) Keep only referenced components that are in allowed root-cause component list
ref_stats_allowed = ref_stats[ref_stats['referenced_component'].isin(allowed_components)].copy()

# 2) For each such X, check component_fault_summary for faults_count>0
# Normalize component_fault_summary column names (cmdb_id, faults_count, earliest_fault_start)
cfs_lookup = cfs.set_index('cmdb_id')[['faults_count','earliest_fault_start']].copy()

def lookup_faults(comp):
    if comp in cfs_lookup.index:
        row = cfs_lookup.loc[comp]
        return int(row['faults_count']), int(row['earliest_fault_start']) if not pd.isna(row['earliest_fault_start']) else pd.NA
    return 0, pd.NA

ref_stats_allowed[['faults_count','earliest_metric_fault_start']] = ref_stats_allowed['referenced_component'].apply(
    lambda c: pd.Series(lookup_faults(c), index=['faults_count','earliest_metric_fault_start'])
)

# is_faulty boolean
ref_stats_allowed['is_faulty'] = ref_stats_allowed['faults_count'] > 0

# Reorder/rename columns for downstream_trace_fault_candidates
downstream_trace_fault_candidates = ref_stats_allowed.rename(columns={'referenced_component':'component'})[
    ['component','is_faulty','faults_count','earliest_metric_fault_start','trace_anomaly_edge_count','earliest_trace_breach_ts']
].copy()

# 3) Sorting: is_faulty True first, then trace_anomaly_edge_count desc, then earliest_metric_fault_start asc (NaNs last)
# For sorting, convert earliest_metric_fault_start to numeric with NaN preserved
downstream_trace_fault_candidates['earliest_metric_fault_start_num'] = pd.to_numeric(downstream_trace_fault_candidates['earliest_metric_fault_start'], errors='coerce')

sorted_candidates = downstream_trace_fault_candidates.sort_values(
    by=['is_faulty','trace_anomaly_edge_count','earliest_metric_fault_start_num'],
    ascending=[False, False, True]
).reset_index(drop=True).drop(columns=['earliest_metric_fault_start_num'])

# Store full mapping table in kernel variable as requested
downstream_trace_fault_candidates = downstream_trace_fault_candidates  # full table

# Prepare compact outputs:
top20_candidates = sorted_candidates.head(20).reset_index(drop=True)
total_distinct_referenced_components = int(downstream_trace_fault_candidates.shape[0])

# Display compact results (variables)
top20_candidates, total_distinct_referenced_components, downstream_trace_fault_candidates.shape

```
Out[6]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing kernel variables if available
try:
    tbl = top20_candidates
    total = total_distinct_referenced_components
except NameError:
    try:
        tbl = downstream_trace_fault_candidates.sort_values(['is_faulty','trace_anomaly_edge_count'], ascending=[False, False]).head(20)
        total = int(downstream_trace_fault_candidates.shape[0])
    except NameError:
        tbl = pd.DataFrame()
        total = 0

def epoch_to_local_iso(e):
    if pd.isna(e):
        return "None"
    try:
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).isoformat()
    except:
        return str(e)

lines = []
lines.append(f"Downstream trace -> metric fault mapping summary (distinct referenced components: {total}):")
lines.append("")
if tbl.empty:
    lines.append("No referenced components found.")
else:
    lines.append("Top referenced components (ranked: faulty first, then by trace-anomaly edge count):")
    for i, row in tbl.reset_index(drop=True).iterrows():
        comp = row.get('component', row.get('component', row.get('component', '')))
        is_faulty = bool(row['is_faulty'])
        faults = int(row['faults_count'])
        earliest_metric = epoch_to_local_iso(row['earliest_metric_fault_start'])
        edges = int(row['trace_anomaly_edge_count'])
        earliest_trace = epoch_to_local_iso(row['earliest_trace_breach_ts'])
        lines.append(
            f" {i+1}. {comp} — is_faulty={is_faulty}, faults_count={faults}, "
            f"earliest_metric_fault_start={earliest_metric}, trace_anomaly_edge_count={edges}, "
            f"earliest_trace_breach_ts={earliest_trace}"
        )

lines.append("")
lines.append("Interpretation:")
lines.append("- Components listed above are referenced by anomalous traces. 'is_faulty=True' means the component also had metric fault segments in the incident window.")
lines.append("- Higher trace_anomaly_edge_count indicates more trace anomaly edges pointing to the component (possible downstream impact).")
lines.append("- Use earliest_metric_fault_start and earliest_trace_breach_ts to correlate which side showed anomalies first.")
summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(  component  is_faulty  faults_count  earliest_metric_fault_start  trace_anomaly_edge_count  earliest_trace_breach_ts
0      MG01       True             2                   1614812400                        36                1614812400
1      MG02       True            12                   1614812400                        36                1614812400
2      IG01       True            12                   1614812400                        12                1614812400
3      IG02       True            14                   1614812400                        12                1614812460
4  Tomcat01       True            19                   1614812400                        12                1614812460
5  Tomcat02       True            21                   1614812400                        12                1614812400
6  Tomcat03       True            23                   1614812400                        12                1614812400
7  Tomcat04       True            21                   1614812400                        12                1614812400, 8, (8, 7))```
```

In [ ]:
# In[7]:

import re
import pandas as pd
import numpy as np

# Reuse existing kernel tables
tia = trace_incident_anomalies.copy()  # trace_incident_anomalies exists in kernel
cfs = component_fault_summary.copy()   # component_fault_summary exists in kernel

# Allowed root-cause component list
allowed_components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                      'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# 1) Extract target component from trace_name patterns trace.to_<TARGET>... or trace.from_<TARGET>...
def extract_target(tn):
    if pd.isna(tn):
        return None
    m = re.match(r'^trace\.(?:to|from)_([A-Za-z0-9_\-]+)', str(tn))
    return m.group(1) if m else None

tia['target_component'] = tia['trace_name'].apply(extract_target)
tia['source_component'] = tia['cmdb_id'].astype(str)

# 2) Keep only edges where both source and target are in allowed list and both are metric-faulty (faults_count>0)
# Prepare lookup for faults_count and earliest_metric_fault_start
cfs_lookup = cfs.set_index('cmdb_id')[['faults_count','earliest_fault_start']].to_dict(orient='index')

def get_fault_info(comp):
    info = cfs_lookup.get(comp)
    if info is None:
        return 0, pd.NA
    return int(info['faults_count']), int(info['earliest_fault_start']) if not pd.isna(info['earliest_fault_start']) else pd.NA

# Apply filters
edges = tia[['source_component','target_component','trace_name','earliest_breach_ts','incident_max','p95']].copy()
edges = edges[edges['target_component'].notna()]

# Filter by allowed components
edges = edges[edges['source_component'].isin(allowed_components) & edges['target_component'].isin(allowed_components)].copy()

# Lookup faults_count for source and target
edges[['source_faults_count','source_earliest_metric_fault_start']] = edges['source_component'].apply(
    lambda c: pd.Series(get_fault_info(c), index=['source_faults_count','source_earliest_metric_fault_start'])
)
edges[['target_faults_count','target_earliest_metric_fault_start']] = edges['target_component'].apply(
    lambda c: pd.Series(get_fault_info(c), index=['target_faults_count','target_earliest_metric_fault_start'])
)

# Keep only where both source and target have faults_count > 0
trace_fault_edges = edges[(edges['source_faults_count'] > 0) & (edges['target_faults_count'] > 0)].copy()

# 3) Compute edge_severity = incident_max / p95 if p95 > 0 else NaN
def safe_sev(row):
    p95 = row['p95']
    imax = row['incident_max']
    if pd.isna(p95) or p95 == 0:
        return np.nan
    return float(imax) / float(p95)

trace_fault_edges['edge_severity'] = trace_fault_edges.apply(safe_sev, axis=1)

# Keep only required columns for the edges table
trace_fault_edges = trace_fault_edges.rename(columns={
    'source_component':'source',
    'target_component':'target',
    'earliest_breach_ts':'earliest_trace_breach_ts'
})[
    ['source','target','trace_name','earliest_trace_breach_ts','incident_max','p95','edge_severity',
     'source_faults_count','target_faults_count','source_earliest_metric_fault_start','target_earliest_metric_fault_start']
].copy()

# 4) Aggregate by target: incoming_faulty_edge_count, earliest_incoming_trace_breach_ts, earliest_metric_fault_start (from component_fault_summary)
agg = trace_fault_edges.groupby('target', sort=True).agg(
    incoming_faulty_edge_count=('source','count'),
    earliest_incoming_trace_breach_ts=('earliest_trace_breach_ts', lambda s: int(s.dropna().min()) if s.dropna().size>0 else pd.NA),
    earliest_metric_fault_start=('target_earliest_metric_fault_start', lambda s: int(pd.Series(s).dropna().min()) if pd.Series(s).dropna().size>0 else pd.NA)
).reset_index()

trace_fault_target_summary = agg.copy()

# 5) Identify terminal targets: target that does NOT appear as a source in trace_fault_edges
sources_set = set(trace_fault_edges['source'].unique())
trace_fault_target_summary['terminal'] = ~trace_fault_target_summary['target'].isin(sources_set)

# 6) Prepare compact result sorted: terminal True first, then incoming_faulty_edge_count desc, then earliest_metric_fault_start asc
# convert earliest_metric_fault_start to numeric for sorting (NaN last)
trace_fault_target_summary['earliest_metric_fault_start_num'] = pd.to_numeric(trace_fault_target_summary['earliest_metric_fault_start'], errors='coerce')
sorted_targets = trace_fault_target_summary.sort_values(
    by=['terminal','incoming_faulty_edge_count','earliest_metric_fault_start_num'],
    ascending=[False, False, True]
).drop(columns=['earliest_metric_fault_start_num']).reset_index(drop=True)

# Final compact output: top 20 rows with required fields
top20_targets = sorted_targets.loc[:, ['target','terminal','incoming_faulty_edge_count','earliest_incoming_trace_breach_ts','earliest_metric_fault_start']].head(20).copy()

# Store full tables in kernel as requested: trace_fault_edges, trace_fault_target_summary
# (they are already assigned)

# Return compact outputs: top20_targets, and shapes for stored tables
top20_targets, trace_fault_edges.shape, trace_fault_target_summary.shape

```
Out[7]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use kernel variables if available
try:
    tbl = top20_targets.copy()
    edges_shape = trace_fault_edges.shape
    targets_shape = trace_fault_target_summary.shape
except NameError:
    # Fallback to previously computed sorted_targets if available
    try:
        tbl = sorted_targets.loc[:, ['target','terminal','incoming_faulty_edge_count','earliest_incoming_trace_breach_ts','earliest_metric_fault_start']].head(20).copy()
        edges_shape = trace_fault_edges.shape
        targets_shape = trace_fault_target_summary.shape
    except NameError:
        tbl = pd.DataFrame()
        edges_shape = (0, 0)
        targets_shape = (0, 0)

def to_local_iso(e):
    if pd.isna(e):
        return "None"
    try:
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).isoformat()
    except:
        return str(e)

lines = []
lines.append("Trace faulty-edge target summary (incident window 2021-03-04 07:00–07:30 UTC+8):")
lines.append("")
lines.append(f"- Total faulty trace edges kept (trace_fault_edges rows): {edges_shape[0]}")
lines.append(f"- Distinct faulty-edge targets found: {targets_shape[0]}")
lines.append("- Stored tables: 'trace_fault_edges' and 'trace_fault_target_summary' in kernel.")
lines.append("")
if tbl.empty:
    lines.append("No qualifying faulty trace edges found between allowed components.")
else:
    lines.append("Top targets (terminal=True means target has no outgoing faulty edges to other faulty components):")
    for i, r in tbl.iterrows():
        tgt = r['target']
        term = bool(r['terminal'])
        cnt = int(r['incoming_faulty_edge_count'])
        earliest_trace = to_local_iso(r['earliest_incoming_trace_breach_ts'])
        earliest_metric = to_local_iso(r['earliest_metric_fault_start'])
        lines.append(f" {i+1}. {tgt} — terminal={term}, incoming_faulty_edge_count={cnt}, earliest_incoming_trace_breach_ts={earliest_trace}, earliest_metric_fault_start={earliest_metric}")
    lines.append("")
    # Overall observation
    # Check if any terminal True in displayed table
    if tbl['terminal'].any():
        lines.append("Observation: Some targets are terminal (they receive faulty edges but do not forward faults further).")
    else:
        lines.append("Observation: None of the top targets are terminal; each also appears as a source for faulty edges to other faulty components.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     target  terminal  incoming_faulty_edge_count  earliest_incoming_trace_breach_ts  earliest_metric_fault_start
0      IG01     False                          12                         1614812400                   1614812400
1      IG02     False                          12                         1614812460                   1614812400
2      MG01     False                          12                         1614812400                   1614812400
3      MG02     False                          12                         1614812460                   1614812400
4  Tomcat01     False                          12                         1614812460                   1614812400
5  Tomcat02     False                          12                         1614812400                   1614812400
6  Tomcat03     False                          12                         1614812400                   1614812400
7  Tomcat04     False                          12                         1614812400                   1614812400, (96, 11), (8, 6))```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables
mfs = metric_fault_segments.copy()         # segments table
cfs = component_fault_summary.copy()      # faults_count and earliest_fault_start

# Ensure expected column names
# metric_fault_segments may use 'cmdb_id' as component column
component_col = 'cmdb_id' if 'cmdb_id' in mfs.columns else ('component' if 'component' in mfs.columns else None)
if component_col is None:
    raise KeyError("metric_fault_segments missing a component id column ('cmdb_id' or 'component').")

# Ensure numeric types
mfs['length_minutes'] = pd.to_numeric(mfs['length_minutes'], errors='coerce').fillna(0).astype(int)
mfs['max_over_p95_ratio'] = pd.to_numeric(mfs['max_over_p95_ratio'], errors='coerce')
mfs['segment_max_value'] = pd.to_numeric(mfs['segment_max_value'], errors='coerce')

# Prepare component fault lookup (faults_count, earliest_metric_fault_start)
cfs_lookup = cfs.set_index('cmdb_id')[['faults_count','earliest_fault_start']].copy()

def lookup_faults(comp):
    if comp in cfs_lookup.index:
        row = cfs_lookup.loc[comp]
        return int(row['faults_count']), (int(row['earliest_fault_start']) if pd.notna(row['earliest_fault_start']) else pd.NA)
    return 0, pd.NA

# 1) Aggregate per-component severity summary
grp = mfs.groupby(component_col, sort=True)

component_severity_summary = grp.agg(
    max_over_p95_ratio = ('max_over_p95_ratio', lambda s: float(s.dropna().max()) if s.dropna().size>0 else np.nan),
    max_segment_length_minutes = ('length_minutes', lambda s: int(s.max()) if s.size>0 else 0),
    count_long_segments = ('length_minutes', lambda s: int((s >= 10).sum()))
).reset_index().rename(columns={component_col:'component'})

# Attach faults_count and earliest_metric_fault_start from component_fault_summary
fc = component_severity_summary['component'].apply(lambda c: pd.Series(lookup_faults(c), index=['faults_count','earliest_metric_fault_start']))
component_severity_summary[['faults_count','earliest_metric_fault_start']] = fc[['faults_count','earliest_metric_fault_start']]

# Reorder columns
component_severity_summary = component_severity_summary[[
    'component','faults_count','earliest_metric_fault_start','max_over_p95_ratio','max_segment_length_minutes','count_long_segments'
]]

# Store full table in kernel variable
component_severity_summary = component_severity_summary

# 2) Return compact top-10 sorted by max_over_p95_ratio desc, then faults_count desc
component_severity_summary_top10 = component_severity_summary.sort_values(
    by=['max_over_p95_ratio','faults_count'],
    ascending=[False, False]
).head(10).reset_index(drop=True)

# 3) For top-ranked component, return up to 10 of its segments sorted by max_over_p95_ratio desc
if not component_severity_summary_top10.empty:
    top_component = component_severity_summary_top10.loc[0, 'component']
    top_component_segments = mfs[mfs[component_col] == top_component].copy()
    # Ensure columns exist and select required columns
    select_cols = []
    # prefer 'kpi_name' column name
    if 'kpi_name' in top_component_segments.columns:
        select_cols.append('kpi_name')
    elif 'kpi' in top_component_segments.columns:
        select_cols.append('kpi')
    else:
        # fallback: include all columns starting with kpi
        kpi_cols = [c for c in top_component_segments.columns if 'kpi' in c]
        select_cols.extend(kpi_cols[:1] if kpi_cols else [])
    # required columns
    for col in ['start_ts','end_ts','length_minutes','p95','p05','segment_max_value','max_over_p95_ratio']:
        if col in top_component_segments.columns:
            select_cols.append(col)
    # sort and take up to 10
    if 'max_over_p95_ratio' in top_component_segments.columns:
        top_component_segments = top_component_segments.sort_values('max_over_p95_ratio', ascending=False)
    top_component_segments = top_component_segments.loc[:, select_cols].head(10).reset_index(drop=True)
else:
    top_component = None
    top_component_segments = pd.DataFrame()

# Keep full intermediate tables in kernel as specified:
# component_severity_summary, top_component_segments

# Display compact outputs (DataFrames)
component_severity_summary_top10, top_component_segments

```
Out[8]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing kernel variables
top10_df = component_severity_summary_top10.copy()
segments_df = top_component_segments.copy()

def epoch_to_iso(e):
    if pd.isna(e):
        return "None"
    try:
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).isoformat()
    except:
        return str(e)

lines = []
lines.append("Component severity summary (top 10) — incident window 2021-03-04 07:00–07:30 (UTC+8):")
lines.append("")
for i, r in top10_df.iterrows():
    comp = r['component']
    faults = int(r['faults_count'])
    earliest_epoch = r['earliest_metric_fault_start']
    earliest_iso = epoch_to_iso(earliest_epoch)
    max_ratio = r['max_over_p95_ratio']
    max_len = int(r['max_segment_length_minutes'])
    long_cnt = int(r['count_long_segments'])
    lines.append(
        f"{i+1}. {comp} | faults_count={faults} | earliest_metric_fault_start={int(earliest_epoch)} ({earliest_iso}) | "
        f"max_over_p95_ratio={max_ratio:.6g} | max_segment_length_minutes={max_len} | count_long_segments={long_cnt}"
    )

lines.append("")
if not segments_df.empty:
    # identify top component name (from earlier step)
    top_comp = top10_df.loc[0, 'component']
    lines.append(f"Top component: {top_comp} — up to 10 segments sorted by max_over_p95_ratio:")
    lines.append("")
    for j, s in segments_df.iterrows():
        kpi = s.get('kpi_name', '(kpi)')
        start = s.get('start_ts', None)
        end = s.get('end_ts', None)
        start_iso = epoch_to_iso(start) if pd.notna(start) else "None"
        end_iso = epoch_to_iso(end) if pd.notna(end) else "None"
        length = int(s.get('length_minutes', 0))
        p95 = s.get('p95', None)
        p05 = s.get('p05', None)
        seg_max = s.get('segment_max_value', None)
        ratio = s.get('max_over_p95_ratio', None)
        lines.append(
            f" - {kpi} | start={int(start)} ({start_iso}) end={int(end)} ({end_iso}) | len={length}min | "
            f"p95={p95:.6g} p05={p05:.6g} | max={seg_max:.6g} | max_over_p95_ratio={ratio:.6g}"
        )
else:
    lines.append("No segments available for the top component.")

lines.append("")
lines.append("Notes:")
lines.append("- max_over_p95_ratio is the largest segment-level ratio (segment_max / p95) for the component.")
lines.append("- count_long_segments counts segments >= 10 minutes (longer, possibly more severe faults).")
lines.append("- All timestamps shown as epoch seconds and local ISO (Asia/Shanghai, UTC+8).")
summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  component  faults_count  earliest_metric_fault_start  max_over_p95_ratio  max_segment_length_minutes  count_long_segments
0   Redis01            33                   1614812400            2.000000                          31                   28
1  apache01            12                   1614812400            1.666667                          31                    7
2   Mysql02             6                   1614812400            1.594937                          31                    5
3      IG01            12                   1614812400            1.312500                          31                    8
4      IG02            14                   1614812400            1.030669                          31                    7
5   Redis02            30                   1614812400            1.000000                          31                   28
6  Tomcat03            23                   1614812400            1.000000                          31                   17
7  Tomcat02            21                   1614812400            1.000000                          31                   17
8  Tomcat04            21                   1614812400            1.000000                          31                   17
9  Tomcat01            19                   1614812400            1.000000                          31                   14,                                             kpi_name    start_ts      end_ts  length_minutes      p95     p05  segment_max_value  max_over_p95_ratio
0      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1614813120  1614814200              19     1.00     1.0               2.00                 2.0
1                            OSLinux-CPU_CPU_CPULoad  1614812400  1614812520               3     1.15     1.0               1.15                 1.0
2  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1614813120  1614814200              19     0.40     0.3               0.40                 1.0
3      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1614812400  1614813000              11     1.00     1.0               1.00                 1.0
4  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1614812400  1614813000              11     0.40     0.3               0.40                 1.0
5             redis-Redis_6379_Redis  (expired_keys)  1614812400  1614814200              31     6.00     6.0               6.00                 1.0
6        redis-Redis_6379_Redis  (connected_clients)  1614812400  1614814200              31   601.00   601.0             601.00                 1.0
7         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  1614813120  1614814200              19     4.00     4.0               4.00                 1.0
8         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  1614812400  1614813000              11     4.00     4.0               4.00                 1.0
9          redis-Redis_6379_Redis  (keyspace_misses)  1614812400  1614814200              31  6938.00  6938.0            6938.00                 1.0)```
```